In [19]:
import pandas as pd
import numpy as np
import joblib

In [20]:
import streamlit as st

In [21]:
user_input = {
    'inning': 1,
    'cum_runs': 80,            # current cumulative runs
    'cum_wickets': 5,          # current wickets lost
    'overs_completed': 12.0,    # overs completed so far
    'target': 0,             # target score
    'batting_team': "Mumbai Indians",
    'bowling_team': "Chennai Super Kings",
    'venue': "Wankhede Stadium"  # assume this maps to a canonical venue (e.g., "Wankhede Stadium")
}

In [22]:
if user_input['overs_completed'] > 0:
    current_run_rate = user_input['cum_runs'] / user_input['overs_completed']
else:
    current_run_rate = 0

# Compute required run rate
remaining_overs = 20 - user_input['overs_completed']
if user_input['inning'] == 2 and remaining_overs > 0:
    required_run_rate = (user_input['target'] - user_input['cum_runs']) / remaining_overs
else:
    required_run_rate = 0

In [23]:
le_team = joblib.load('/content/le_team123.pkl')
le_venue = joblib.load('/content/le_venue123.pkl')

# Load the final trained model (e.g., a Random Forest model)
model = joblib.load('/content/final_rf_model.pkl')

In [24]:
if user_input['venue'] in le_venue.classes_:
    venue_canonical_encoded = le_venue.transform([user_input['venue']])[0]
else:
    print(f"Warning: Venue '{user_input['venue']}' not found in trained data. Assigning default value.")
    venue_canonical_encoded = -1  # Assigning a default value


In [25]:
# Get all existing venue labels
existing_venues = list(le_venue.classes_)

# If the new venue is not in the existing list, add it
if user_input['venue'] not in existing_venues:
    existing_venues.append(user_input['venue'])
    le_venue.classes_ = np.array(existing_venues)

# Now transform the venue
venue_canonical_encoded = le_venue.transform([user_input['venue']])[0]

In [26]:
batting_team_encoded = le_team.transform([user_input['batting_team']])[0]
bowling_team_encoded = le_team.transform([user_input['bowling_team']])[0]

# Encode the venue using the venue encoder
venue_canonical_encoded = le_venue.transform([user_input['venue']])[0]



In [28]:
# Calculate additional required features
total_overs = 20  # Adjust if it's an ODI match (50 overs)
remaining_overs = max(total_overs - user_input["overs_completed"], 1)  # Avoid division by zero

current_run_rate = user_input["cum_runs"] / max(user_input["overs_completed"], 1)  # Avoid division by zero
required_run_rate = (user_input["target"] - user_input["cum_runs"]) / remaining_overs

# Create input DataFrame with correct column names
input_df = pd.DataFrame([{
    "inning": user_input["inning"],
    "cum_runs": user_input["cum_runs"],
    "cum_wickets": user_input["cum_wickets"],
    "overs_completed": user_input["overs_completed"],
    "target": user_input["target"],
    "batting_team": batting_team_encoded,  # Match training data
    "bowling_team": bowling_team_encoded,  # Match training data
    "venue_encoded": venue_canonical_encoded,  # Match training data
    "current_run_rate": current_run_rate,  # Added missing feature
    "required_run_rate": required_run_rate  # Added missing feature
}])

# Ensure the column order matches the training data
input_df = input_df[model.feature_names_in_]

# Make predictions
prediction = model.predict(input_df)[0]
predicted_probabilities = model.predict_proba(input_df)[0]

# Map Prediction to Team Names
predicted_winner = user_input["batting_team"] if prediction == 1 else user_input["bowling_team"]

print("Predicted Winner:", predicted_winner)
print("Prediction Probabilities (Loss, Win):", predicted_probabilities)


Predicted Winner: Chennai Super Kings
Prediction Probabilities (Loss, Win): [0.97 0.03]
